# DSSAT Successive Converter - Test Run

This notebook runs the DSSAT successive converter. Simulations are grouped from `SimUnitList` by `idPoint`, `idMangt`, and `idOption`, ordered by `StartYear` and `StartDay`, then executed with DSSAT native sequence mode (`Q`) using a generated `.SQX` file and `DSSBatch.v47`.

When two crop simulations have a calendar gap, the converter inserts a DSSAT `FA` / `FALLOW` bare-soil rotation between them, using the same inter-crop logic as the STICS successive converter.


## 1. Locate the Repository and Import Modules

In [1]:
import os
import sys
from datetime import timedelta
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
repo_root = next(
    (
        path
        for path in candidates
        if (path / "tests" / "data").exists() and (path / "src" / "modfilegen").exists()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError(f"Could not locate the ModFileGen repository from cwd={cwd}.")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from modfilegen import GlobalVariables
from modfilegen.Converter.DssatConverter.dssatsuccessiveconverter import (
    build_successive_groups,
    dssat_sequence_years,
    main,
    row_end_date,
    row_start_date,
)
from modfilegen.Converter.DssatConverter.dssatconverter import fetch_data_from_sqlite

print(f"Kernel cwd: {cwd}")
print(f"Repo root: {repo_root}")

Kernel cwd: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/notebooks
Repo root: /mnt/d/Mes Donnees/TCMP/github/ModFileGen


## 2. Configure Paths

In [2]:
data_dir = repo_root / "tests" / "data"
output_dir = repo_root / "tests" / "output_dssat_successive"
temp_dir = output_dir / "temp"

master_input_db = data_dir / "MasterInput.db"
models_dict_db = data_dir / "ModelsDictionaryArise.db"
cultivars_folder = data_dir / "cultivars" / "dssat"

output_dir.mkdir(parents=True, exist_ok=True)
temp_dir.mkdir(parents=True, exist_ok=True)

n_threads = 1
# dt=0 keeps generated sequence folders so ITSA1301.SQX and DSSBatch.v47 can be inspected.
# Set dt=1 to clean temporary folders after each successful group.
dt = 0

print(f"Master Input DB exists: {master_input_db.exists()} -> {master_input_db}")
print(f"Models Dict DB exists: {models_dict_db.exists()} -> {models_dict_db}")
print(f"Cultivars folder exists: {cultivars_folder.exists()} -> {cultivars_folder}")
print(f"Output directory: {output_dir}")
print(f"Temp directory: {temp_dir}")

Master Input DB exists: True -> /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/data/MasterInput.db
Models Dict DB exists: True -> /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/data/ModelsDictionaryArise.db
Cultivars folder exists: True -> /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/data/cultivars/dssat
Output directory: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat_successive
Temp directory: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat_successive/temp


## 3. Preview Successive Groups

This checks how simulations will be grouped, how many DSSAT sequence years each group will request, and where bare-soil fallow rotations will be inserted between crops.


In [3]:
rows = fetch_data_from_sqlite(str(master_input_db))
groups = build_successive_groups(rows)

print(f"Total simulations: {len(rows)}")
print(f"DSSAT successive groups: {len(groups)}")

for group_index, group in enumerate(groups[:5], start=1):
    key = (group[0]["idPoint"], group[0]["idMangt"], group[0]["idOption"])
    bare_soil_periods = []
    for index, row in enumerate(group[:-1]):
        next_row = group[index + 1]
        bare_start = row_end_date(row) + timedelta(days=1)
        bare_end = row_start_date(next_row) - timedelta(days=1)
        if bare_start <= bare_end:
            bare_soil_periods.append((row, next_row, bare_start, bare_end))

    print()
    print(f"Group {group_index}: {key}")
    print(f"  crop simulations: {len(group)}")
    print(f"  effective DSSAT rotations: {len(group) + len(bare_soil_periods)}")
    print(f"  bare-soil rotations: {len(bare_soil_periods)}")
    print(f"  DSSAT NYERS: {dssat_sequence_years(group)}")
    for row in group[:5]:
        print(
            f"  crop {row['idsim']}: "
            f"start={row['StartYear']}-{row['StartDay']} "
            f"end={row['EndYear']}-{row['EndDay']}"
        )
    for row, next_row, bare_start, bare_end in bare_soil_periods[:5]:
        print(
            f"  bare_soil {row['idsim']}__intercrop__{next_row['idsim']}: "
            f"start={bare_start.year}-{bare_start.timetuple().tm_yday} "
            f"end={bare_end.year}-{bare_end.timetuple().tm_yday}"
        )


Total simulations: 9
DSSAT successive groups: 3

Group 1: ('5.925_6.025', 'Mgt1M0_135', 2)
  crop simulations: 3
  effective DSSAT rotations: 5
  bare-soil rotations: 2
  DSSAT NYERS: 3
  crop 5.925_6.025_2000_Mgt1M0_135_2: start=2000-105 end=2000-335
  crop 5.925_6.025_2001_Mgt1M0_135_2: start=2001-105 end=2001-335
  crop 5.925_6.025_2002_Mgt1M0_135_2: start=2002-105 end=2002-335
  bare_soil 5.925_6.025_2000_Mgt1M0_135_2__intercrop__5.925_6.025_2001_Mgt1M0_135_2: start=2000-336 end=2001-104
  bare_soil 5.925_6.025_2001_Mgt1M0_135_2__intercrop__5.925_6.025_2002_Mgt1M0_135_2: start=2001-336 end=2002-104

Group 2: ('5.925_6.025', 'Mgt1M0_150', 2)
  crop simulations: 3
  effective DSSAT rotations: 5
  bare-soil rotations: 2
  DSSAT NYERS: 3
  crop 5.925_6.025_2000_Mgt1M0_150_2: start=2000-120 end=2000-350
  crop 5.925_6.025_2001_Mgt1M0_150_2: start=2001-120 end=2001-350
  crop 5.925_6.025_2002_Mgt1M0_150_2: start=2002-120 end=2002-350
  bare_soil 5.925_6.025_2000_Mgt1M0_150_2__intercrop__

## 4. Set Global Variables

In [4]:
GlobalVariables["dbMasterInput"] = str(master_input_db)
GlobalVariables["dbModelsDictionary"] = str(models_dict_db)
GlobalVariables["directorypath"] = str(output_dir)
GlobalVariables["pltfolder"] = str(cultivars_folder)
GlobalVariables["nthreads"] = n_threads
GlobalVariables["dt"] = dt
GlobalVariables["parts"] = 1
GlobalVariables["tempDir"] = str(temp_dir)

print("GlobalVariables configured:")
for key, value in GlobalVariables.items():
    print(f"  {key}: {value}")

GlobalVariables configured:
  storeNumMinSimu: 0
  storeNumMaxSimu: 0
  storeKeyDataN: 0
  dbMasterInput: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/data/MasterInput.db
  dbModelsDictionary: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/data/ModelsDictionaryArise.db
  dbCelsius: 
  dt: 0
  ori_MI: 
  parts: 1
  tempDir: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat_successive/temp
  package: 
  thirdyear: 0
  directorypath: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat_successive
  pltfolder: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/data/cultivars/dssat
  nthreads: 1


## 5. Run the Successive DSSAT Converter

Each group is converted to a native DSSAT sequence experiment and run with `dssat Q DSSBatch.v47`.

In [5]:
print("Starting DSSAT successive conversion...")
print("=" * 60)

result_path = main()

print("=" * 60)
print(f"Result CSV: {result_path}")

Starting DSSAT successive conversion...
Indexes created successfully!
Total simulations to process: 9
DSSAT successive groups: 3
Parallel workers: 1
Processing DSSAT successive group 5.925_6.025__Mgt1M0_135__2 with 3 simulation(s)


Processing DSSAT successive group 5.925_6.025__Mgt1M0_150__2 with 3 simulation(s)
Processing DSSAT successive group 5.925_6.025__Mgt1M0_155__2 with 3 simulation(s)
Results saved to /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat_successive/01064a4a-3a0a-4c4f-8bcb-5ad2d6c361d3_dssat_successive.csv
DSSAT successive total time, 2.981915235519409
Result CSV: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat_successive/01064a4a-3a0a-4c4f-8bcb-5ad2d6c361d3_dssat_successive.csv


## 6. Check Generated Files

In [6]:
from itertools import islice
import pandas as pd

csv_files = sorted(output_dir.glob("*_dssat_successive.csv"), key=lambda path: path.stat().st_mtime, reverse=True)
print(f"Successive result CSV files: {len(csv_files)}")
for path in csv_files[:5]:
    print(f"  {path.name}")

sequence_files = sorted(temp_dir.glob("*/ITSA1301.SQX"))
batch_files = sorted(temp_dir.glob("*/DSSBatch.v47"))
print()
print(f"Generated SQX files: {len(sequence_files)}")
for path in islice(sequence_files, 5):
    print(f"  {path}")
print()
print(f"Generated DSSBatch files: {len(batch_files)}")
for path in islice(batch_files, 5):
    print(f"  {path}")

if result_path:
    result_df = pd.read_csv(result_path)
    print()
    print(f"Rows in result CSV: {len(result_df)}")
    display(result_df.head())


Successive result CSV files: 2
  816d2b78-c89b-4159-acfa-a951d7fbdcc7_dssat_successive.csv
  f2deec7e-d254-4cfd-86c7-610e9cbc86d0_dssat_successive.csv

Generated SQX files: 3
  /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat_successive/temp/5.925_6.025__Mgt1M0_135__2/ITSA1301.SQX
  /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat_successive/temp/5.925_6.025__Mgt1M0_150__2/ITSA1301.SQX
  /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat_successive/temp/5.925_6.025__Mgt1M0_155__2/ITSA1301.SQX

Generated DSSBatch files: 3
  /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat_successive/temp/5.925_6.025__Mgt1M0_135__2/DSSBatch.v47
  /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat_successive/temp/5.925_6.025__Mgt1M0_150__2/DSSBatch.v47
  /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat_successive/temp/5.925_6.025__Mgt1M0_155__2/DSSBatch.v47

Rows in result CSV: 9


,Model,Idsim,Texte,Planting,Emergence,Ant,Mat,HDAT,DWAP,Biom_ma,...,SRADA,DAYLA,CO2A,PRCP,ETCP,CumE,Transp,lat,lon,time
0,Dssat,5.925_6.025_2000_Mgt1M0_135_2,NaN,2000135.0,2000140.0,2000210.0,2000267.0,2000335.0,-99.0,9426.0,...,14.8,12.1,369.6,1877.5,670.4,412.2,259.7,5.925,6.025,2000
1,Dssat,5.925_6.025_2001_Mgt1M0_135_2,NaN,2001135.0,2001140.0,2001208.0,2001265.0,2001335.0,-99.0,9955.0,...,15.0,12.1,371.2,1476.0,653.6,381.7,273.2,5.925,6.025,2001
2,Dssat,5.925_6.025_2002_Mgt1M0_135_2,NaN,2002135.0,2002140.0,2002208.0,2002264.0,2002335.0,-99.0,8891.0,...,14.6,12.1,373.3,1908.1,657.8,406.4,252.6,5.925,6.025,2002
3,Dssat,5.925_6.025_2000_Mgt1M0_150_2,NaN,2000150.0,2000155.0,2000226.0,2000283.0,2000350.0,-99.0,10020.0,...,15.0,12.0,369.7,1758.8,644.7,368.0,277.7,5.925,6.025,2000
4,Dssat,5.925_6.025_2001_Mgt1M0_150_2,NaN,2001150.0,2001155.0,2001225.0,2001281.0,2001350.0,-99.0,10672.0,...,15.1,12.0,371.3,1346.7,622.1,336.4,286.4,5.925,6.025,2001


## 7. Inspect One DSSAT Sequence

In [7]:
if sequence_files:
    sqx_path = sequence_files[0]
    batch_path = sqx_path.parent / "DSSBatch.v47"
    print(f"SQX: {sqx_path}")
    print()
    print("=" * 20 + " DSSBatch.v47 " + "=" * 20)
    print(batch_path.read_text()[:2000])
    print()
    print("=" * 20 + " ITSA1301.SQX " + "=" * 20)
    print(sqx_path.read_text()[:5000])
else:
    print("No sequence files found. If dt=1, temporary sequence folders were cleaned after execution.")


SQX: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat_successive/temp/5.925_6.025__Mgt1M0_135__2/ITSA1301.SQX

==================== DSSBatch.v47 ====================

$BATCH(EXPERIMENT)
@FILEX                                                                                        TRTNO     RP     SQ     OP     CO
ITSA1301.SQX                                                                                      1      1      1      1      0
ITSA1301.SQX                                                                                      1      1      2      1      0
ITSA1301.SQX                                                                                      1      1      3      1      0
ITSA1301.SQX                                                                                      1      1      4      1      0
ITSA1301.SQX                                                                                      1      1      5      1      0


==================== ITSA1301.SQ

## 8. Plot DSSAT Daily Sequence Variables

This mirrors the STICS successive notebook plots with DSSAT output names:

- STICS `resmes` -> DSSAT `SWTD` from `SoilWat.OUT`.
- STICS `azomes` -> DSSAT `NIAD` from `SoilNi.OUT`, when nitrogen daily output is enabled.
- STICS `Qles` -> DSSAT `NLCC` from `SoilNi.OUT`, when nitrogen daily output is enabled. `DRNC` from `SoilWat.OUT` is plotted as the drainage-water driver, not as N leaching.
- STICS `HR(1)` / `HR(2)` -> DSSAT `SW1D` / `SW2D` from `SoilWat.OUT`.
- STICS `Chumt` -> DSSAT `SCTD` and `SOMCT` from `SoilOrg.OUT`.

If `SoilNi.OUT` is not generated, daily mineral N and daily N leaching cannot be plotted from the current DSSAT output folder; the notebook also reads seasonal `NLCM` and `NIAM` from `Summary.OUT` when available.


In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DSSAT_DAILY_FILES = [
    "SoilWat.OUT",
    "SoilNi.OUT",
    "SoilOrg.OUT",
    "Mulch.OUT",
    "ET.OUT",
    "PlantGro.OUT",
    "PlantN.OUT",
]

DSSAT_DAILY_VARIABLES = [
    {
        "name": "SWTD",
        "ylabel": "SWTD (mm)",
        "title": "Soil Water Reserve - STICS resmes equivalent",
        "color": "tab:blue",
    },
    {
        "name": "NIAD",
        "ylabel": "NIAD (kg N/ha)",
        "title": "Total Mineral Nitrogen - STICS azomes equivalent",
        "color": "tab:green",
    },
    {
        "name": "NLCC",
        "ylabel": "NLCC (kg N/ha)",
        "title": "Cumulative N Leaching - STICS Qles equivalent",
        "color": "tab:purple",
    },
    {
        "name": "DRNC",
        "ylabel": "DRNC (mm)",
        "title": "Cumulative Drainage Water",
        "color": "tab:cyan",
    },
    {
        "name": "SW1D",
        "ylabel": "SW (cm3/cm3)",
        "title": "Surface Soil Water - STICS HR(1)/HR(2) equivalent",
        "color": "tab:red",
        "additional_vars": ["SW2D"],
    },
    {
        "name": "SCTD",
        "ylabel": "Soil C",
        "title": "Soil Organic Carbon - STICS Chumt analogue",
        "color": "tab:brown",
        "additional_vars": ["SOMCT"],
    },
]

DSSAT_SUMMARY_VARIABLES = [
    {
        "name": "NLCM",
        "ylabel": "NLCM (kg N/ha)",
        "title": "Seasonal N Leaching",
        "color": "tab:purple",
    },
    {
        "name": "NIAM",
        "ylabel": "NIAM (kg N/ha)",
        "title": "Seasonal Inorganic N at End of Period",
        "color": "tab:green",
    },
]


def parse_dssat_yyyydoy(value):
    if pd.isna(value):
        return pd.NaT
    try:
        ivalue = int(float(value))
    except (TypeError, ValueError):
        return pd.NaT
    if ivalue <= 0:
        return pd.NaT
    year = ivalue // 1000
    doy = ivalue % 1000
    if year <= 0 or doy <= 0:
        return pd.NaT
    return pd.Timestamp(year=year, month=1, day=1) + pd.Timedelta(days=doy - 1)


def make_date_from_year_doy(df):
    years = pd.to_numeric(df["YEAR"], errors="coerce")
    doys = pd.to_numeric(df["DOY"], errors="coerce")
    return pd.to_datetime(
        years.astype("Int64").astype(str) + doys.astype("Int64").astype(str).str.zfill(3),
        format="%Y%j",
        errors="coerce",
    )


def parse_dssat_out_table(path):
    path = Path(path)
    records = []
    header = None
    current_run = None
    current_trno = None
    current_treatment = None

    for raw_line in path.read_text(errors="ignore").splitlines():
        stripped = raw_line.strip()
        if not stripped:
            continue

        run_match = re.match(r"^\*RUN\s+(\d+)", stripped)
        if run_match:
            current_run = int(run_match.group(1))
            continue

        treatment_match = re.match(r"^TREATMENT\s+(\d+)\s*:\s*(.*)$", stripped)
        if treatment_match:
            current_trno = int(treatment_match.group(1))
            current_treatment = treatment_match.group(2).strip()
            continue

        if stripped.startswith("@"):
            header = stripped[1:].split()
            continue

        if header is None or not re.match(r"^-?\d", stripped):
            continue

        values = stripped.split()
        if len(values) < len(header):
            continue

        row = dict(zip(header, values[: len(header)]))
        row["RUN"] = current_run
        row["TRNO"] = current_trno
        row["treatment_name"] = current_treatment
        row["source_file"] = path.name
        records.append(row)

    df = pd.DataFrame(records)
    if df.empty:
        return df

    for column in df.columns:
        if column not in {"CR", "MODEL...", "EXNAME..", "TNAM.....................", "FNAM....", "WSTA....", "SOIL_ID...", "treatment_name", "source_file"}:
            converted = pd.to_numeric(df[column], errors="coerce")
            if converted.notna().sum() > 0:
                df[column] = converted

    numeric_columns = df.select_dtypes(include=["number"]).columns
    df[numeric_columns] = df[numeric_columns].replace(-99, np.nan)

    if {"YEAR", "DOY"}.issubset(df.columns):
        df["date"] = make_date_from_year_doy(df)

    return df


def load_dssat_summary(sequence_dir):
    summary_path = Path(sequence_dir) / "Summary.OUT"
    if not summary_path.exists():
        return pd.DataFrame()

    summary = parse_dssat_out_table(summary_path)
    if summary.empty:
        return summary

    for date_column in ["SDAT", "PDAT", "EDAT", "ADAT", "MDAT", "HDAT"]:
        if date_column in summary.columns:
            summary[f"{date_column}_date"] = summary[date_column].map(parse_dssat_yyyydoy)

    summary["period_start"] = summary.get("SDAT_date", pd.NaT)
    summary["period_end"] = summary.get("HDAT_date", pd.NaT)
    summary["period_type"] = np.where(summary.get("CR", "") == "FA", "fallow", "crop")
    return summary


def assign_dssat_periods(daily, summary):
    daily = daily.copy()
    daily["period_run"] = np.nan
    daily["period_cr"] = None
    daily["period_type"] = None
    daily["period_label"] = None

    if summary.empty or "period_start" not in summary.columns or "period_end" not in summary.columns:
        return daily

    for _, period in summary.dropna(subset=["period_start", "period_end"]).iterrows():
        mask = daily["date"].between(period["period_start"], period["period_end"], inclusive="both")
        run_number = period.get("RUNNO", np.nan)
        crop_code = period.get("CR", "")
        period_type = period.get("period_type", "crop")
        daily.loc[mask, "period_run"] = run_number
        daily.loc[mask, "period_cr"] = crop_code
        daily.loc[mask, "period_type"] = period_type
        daily.loc[mask, "period_label"] = f"RUN {run_number:g} {crop_code}" if pd.notna(run_number) else crop_code

    return daily


def load_dssat_daily_sequence(sequence_dir):
    sequence_dir = Path(sequence_dir)
    frames = []
    for filename in DSSAT_DAILY_FILES:
        path = sequence_dir / filename
        if not path.exists():
            continue
        frame = parse_dssat_out_table(path)
        if frame.empty:
            continue
        frames.append((path.stem, frame))

    if not frames:
        return pd.DataFrame(), load_dssat_summary(sequence_dir)

    combined = None
    key_cols = ["YEAR", "DOY"]
    excluded_from_merge = set(key_cols + ["date", "source_file", "RUN", "TRNO", "treatment_name", "DAS"])

    for stem, frame in frames:
        frame = frame.drop_duplicates(subset=key_cols, keep="last").copy()
        if combined is None:
            combined = frame.copy()
            continue

        value_cols = [column for column in frame.columns if column not in excluded_from_merge]
        merge_frame = frame[key_cols + value_cols].copy()
        rename_map = {column: f"{column}_{stem}" for column in value_cols if column in combined.columns}
        merge_frame = merge_frame.rename(columns=rename_map)
        combined = combined.merge(merge_frame, on=key_cols, how="outer")

    combined = combined.sort_values(key_cols).reset_index(drop=True)
    combined["date"] = make_date_from_year_doy(combined)

    summary = load_dssat_summary(sequence_dir)
    combined = assign_dssat_periods(combined, summary)
    return combined, summary


def find_sequence_dirs(temp_dir):
    temp_dir = Path(temp_dir)
    return sorted(path.parent for path in temp_dir.glob("*/ITSA1301.SQX"))


def add_period_shading(ax, summary):
    if summary.empty:
        return
    for _, period in summary.dropna(subset=["period_start", "period_end"]).iterrows():
        color = "0.85" if period.get("period_type") == "fallow" else "tab:olive"
        alpha = 0.18 if period.get("period_type") == "fallow" else 0.06
        ax.axvspan(period["period_start"], period["period_end"], color=color, alpha=alpha, linewidth=0)

    starts = summary.dropna(subset=["period_start"]).sort_values("period_start")["period_start"].tolist()
    for start in starts[1:]:
        ax.axvline(start, color="black", linestyle="--", linewidth=0.8, alpha=0.45)


def plot_dssat_daily_variables(daily, summary, sequence_name, variables=DSSAT_DAILY_VARIABLES):
    available_configs = []
    skipped = []
    for config in variables:
        names = [config["name"]] + config.get("additional_vars", [])
        present = [name for name in names if name in daily.columns]
        if present:
            available_configs.append((config, present))
        else:
            skipped.append(config["name"])

    if skipped:
        print("Skipped missing daily DSSAT variables:", ", ".join(skipped))
    if not available_configs:
        print("No configured DSSAT daily variables are available in this sequence folder.")
        return

    fig, axes = plt.subplots(len(available_configs), 1, figsize=(14, 3.2 * len(available_configs)), sharex=True)
    axes = np.atleast_1d(axes)

    for ax, (config, names) in zip(axes, available_configs):
        add_period_shading(ax, summary)
        for index, name in enumerate(names):
            color = config.get("color") if index == 0 else None
            values = pd.to_numeric(daily[name], errors="coerce")
            ax.plot(daily["date"], values, label=name, linewidth=1.6, color=color)
        ax.set_title(config["title"])
        ax.set_ylabel(config["ylabel"])
        ax.grid(alpha=0.25)
        ax.legend(loc="best")

    axes[-1].set_xlabel("Date")
    fig.suptitle(f"DSSAT successive sequence: {sequence_name}", y=1.01, fontsize=14)
    fig.tight_layout()
    plt.show()


def plot_dssat_summary_variables(summary, sequence_name, variables=DSSAT_SUMMARY_VARIABLES):
    if summary.empty:
        print("No Summary.OUT data found.")
        return

    available_configs = [config for config in variables if config["name"] in summary.columns]
    if not available_configs:
        print("No configured DSSAT summary variables are available.")
        return

    plot_summary = summary.copy()
    plot_summary["xdate"] = plot_summary["period_end"].fillna(plot_summary["period_start"])
    labels = [f"{int(row.RUNNO)} {row.CR}" if pd.notna(row.RUNNO) else str(row.CR) for row in plot_summary.itertuples()]

    fig, axes = plt.subplots(len(available_configs), 1, figsize=(14, 3.0 * len(available_configs)), sharex=True)
    axes = np.atleast_1d(axes)

    for ax, config in zip(axes, available_configs):
        values = pd.to_numeric(plot_summary[config["name"]], errors="coerce")
        if values.notna().sum() == 0:
            ax.text(0.5, 0.5, f"{config['name']} is present but all values are missing (-99).", ha="center", va="center", transform=ax.transAxes)
            ax.set_axis_off()
            continue
        ax.bar(plot_summary["xdate"], values, width=20, color=config.get("color"), alpha=0.75)
        for xvalue, yvalue, label in zip(plot_summary["xdate"], values, labels):
            if pd.notna(xvalue) and pd.notna(yvalue):
                ax.text(xvalue, yvalue, label, rotation=90, va="bottom", ha="center", fontsize=8)
        ax.set_title(config["title"])
        ax.set_ylabel(config["ylabel"])
        ax.grid(axis="y", alpha=0.25)

    axes[-1].set_xlabel("Period end date")
    fig.suptitle(f"DSSAT seasonal summary: {sequence_name}", y=1.01, fontsize=14)
    fig.tight_layout()
    plt.show()


sequence_dirs = find_sequence_dirs(temp_dir)
print(f"DSSAT sequence folders found: {len(sequence_dirs)}")

if sequence_dirs:
    selected_sequence_dir = sequence_dirs[0]
    daily_df, summary_df = load_dssat_daily_sequence(selected_sequence_dir)
    print(f"Selected sequence: {selected_sequence_dir.name}")
    print(f"Daily rows: {len(daily_df)}")
    print(f"Summary rows: {len(summary_df)}")
    display(daily_df.head())
    display(summary_df[[column for column in ["RUNNO", "CR", "period_start", "period_end", "PRCM", "DRCM", "NLCM", "NIAM"] if column in summary_df.columns]])

    plot_dssat_daily_variables(daily_df, summary_df, selected_sequence_dir.name)
    plot_dssat_summary_variables(summary_df, selected_sequence_dir.name)
else:
    print("No DSSAT sequence folder found. Run the converter with dt=0, or point selected_sequence_dir to a generated sequence folder.")


## 9. Compare DSSAT Treatments

This section overlays the same DSSAT variables across all generated sequence folders. Missing variables are skipped, which is expected for `NIAD` and `NLCC` unless `SoilNi.OUT` has been generated.


In [ ]:
def load_all_dssat_sequences(sequence_dirs):
    loaded = {}
    summaries = {}
    for sequence_dir in sequence_dirs:
        daily, summary = load_dssat_daily_sequence(sequence_dir)
        if daily.empty:
            continue
        loaded[Path(sequence_dir).name] = daily
        summaries[Path(sequence_dir).name] = summary
    return loaded, summaries


def plot_dssat_treatment_comparison(all_daily, variables=DSSAT_DAILY_VARIABLES):
    if not all_daily:
        print("No DSSAT daily sequence data loaded for comparison.")
        return

    for config in variables:
        names = [config["name"]] + config.get("additional_vars", [])
        present_in_any = [name for name in names if any(name in daily.columns for daily in all_daily.values())]
        if not present_in_any:
            print(f"Skipped {config['name']}: not available in any sequence folder.")
            continue

        fig, ax = plt.subplots(figsize=(14, 4.5))
        for sequence_name, daily in all_daily.items():
            for name in present_in_any:
                if name not in daily.columns:
                    continue
                label = sequence_name if len(present_in_any) == 1 else f"{sequence_name} - {name}"
                values = pd.to_numeric(daily[name], errors="coerce")
                ax.plot(daily["date"], values, label=label, linewidth=1.2, alpha=0.85)

        ax.set_title(config["title"])
        ax.set_ylabel(config["ylabel"])
        ax.set_xlabel("Date")
        ax.grid(alpha=0.25)
        ax.legend(loc="best", fontsize=8)
        fig.tight_layout()
        plt.show()


all_daily_sequences, all_summary_sequences = load_all_dssat_sequences(sequence_dirs)
print(f"Loaded DSSAT sequences for comparison: {len(all_daily_sequences)}")
plot_dssat_treatment_comparison(all_daily_sequences)
